# Improve document understanding with vision and Code Interpreter

Multimodal models are already strong at understanding documents, charts, forms, and mixed-layout images. With a few practical configuration choices, you can make outputs even more faithful on dense pages, preserve more structure in transcriptions, and give the model a richer visual workflow when you want extra inspection.

In this notebook, we'll walk through a set of reusable patterns for getting the best results from the Responses API on document and multimodal tasks:

- Increase image detail when pages contain small text or crowded layouts.
- Ask for fuller transcriptions when fidelity matters more than brevity.
- Return bounding boxes in a stable normalized format.
- Use Code Interpreter, or lightweight visual tools in restricted environments, for harder visual QA.
- Increase reasoning for chart, table, and form QA when the answer depends on multiple parts of the page.

## Prerequisites

- Python 3.10+
- `OPENAI_API_KEY` set in your environment

## Setup

Install the minimum dependencies if needed. The API calls in this notebook are opt-in so you can read the notebook top-to-bottom without spending tokens until you are ready.


In [ ]:
# Uncomment if your environment is missing these packages.
# %pip install -q openai pillow pydantic


In [ ]:
import os
from pathlib import Path

from openai import OpenAI

# Set this to True when you are ready to make live API calls.
RUN_LIVE_API = False

# Replace these with your own assets.
DOCUMENT_IMAGE_URL = "https://example.com/dense-document.png"
LOCAL_VISUAL_ASSET = Path("YOUR_LOCAL_VISUAL_ASSET.png")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if RUN_LIVE_API and not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY before enabling live API calls.")

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

print({
    "run_live_api": RUN_LIVE_API,
    "has_openai_api_key": bool(OPENAI_API_KEY),
    "document_image_url": DOCUMENT_IMAGE_URL,
    "local_visual_asset": str(LOCAL_VISUAL_ASSET),
})


## Pattern 1: Raise image detail for dense documents

When a page contains small text, tight tables, handwriting, or many visual regions, increase image detail. The image input `detail` parameter supports `low`, `high`, and `auto` in the public API docs. Start with `auto` for general use, and move to `high` when the page is dense or the model is missing small text.

More generally, use the highest documented detail setting available in the environment where you deploy. For the public Responses API today, that means moving from `auto` to `high` on difficult pages.

Use this pattern for:

- dense OCR
- handwritten notes
- forms with small labels
- chart labels
- mixed text-and-image layouts


In [ ]:
if RUN_LIVE_API:
    detail_response = client.responses.create(
        model="gpt-5",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Transcribe this page into markdown. Preserve headings, lists, "
                            "and table structure. If a token is unclear, mark it with [?] "
                            "instead of guessing."
                        ),
                    },
                    {
                        "type": "input_image",
                        "image_url": DOCUMENT_IMAGE_URL,
                        "detail": "high",
                    },
                ],
            }
        ],
    )
    print(detail_response.output_text)
else:
    print("Set RUN_LIVE_API = True to run the detail experiment.")


## Pattern 2: Raise verbosity when you want faithful transcription

For GPT-5 series models, the `verbosity` parameter is a useful control when the model tends to compress structured content into a shorter summary. Higher verbosity is especially useful when you want the output to stay closer to the page layout.

Use higher verbosity when:

- the output is stopping early
- the model is collapsing tables into prose
- you want richer audit trails
- you are comparing transcription fidelity across models


In [ ]:
if RUN_LIVE_API:
    verbose_response = client.responses.create(
        model="gpt-5",
        verbosity="high",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Transcribe this page into markdown. Keep section hierarchy, "
                            "line breaks, and table rows as faithfully as possible."
                        ),
                    },
                    {
                        "type": "input_image",
                        "image_url": DOCUMENT_IMAGE_URL,
                        "detail": "high",
                    },
                ],
            }
        ],
    )
    print(verbose_response.output_text)
else:
    print("Set RUN_LIVE_API = True to run the verbosity experiment.")


## Pattern 3: Ask for bounding boxes in a stable format

For localization tasks, use a consistent box format like `[x_min, y_min, x_max, y_max]`. A practical convention is to request discrete normalized coordinates between `0` and `999`, with `(0, 0)` at the top-left corner.

This format is easy to post-process for:

- cropping regions
- drawing overlays
- building review tools
- routing specific page regions into downstream OCR or extraction pipelines


In [ ]:
import json

bbox_prompt = """
Find every table in this page.

Return JSON with this schema:
[
  {
    \"label\": \"table\",
    \"bbox\": [x_min, y_min, x_max, y_max]
  }
]

Use discrete normalized coordinates between 0 and 999.
"""

if RUN_LIVE_API:
    bbox_response = client.responses.create(
        model="gpt-5",
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": bbox_prompt},
                    {
                        "type": "input_image",
                        "image_url": DOCUMENT_IMAGE_URL,
                        "detail": "high",
                    },
                ],
            }
        ],
    )
    bboxes = json.loads(bbox_response.output_text)
    print(json.dumps(bboxes, indent=2))
else:
    print("Set RUN_LIVE_API = True to run the bounding-box experiment.")


## Pattern 4: Use Code Interpreter for hard visual QA

Some visual tasks benefit from more than one pass over the input. Code Interpreter helps because the model can crop, zoom, rotate, and inspect the file programmatically before answering.

This is useful for:

- charts with tiny labels
- skewed or rotated scans
- documents with hard-to-read footnotes
- mixed-layout pages where the answer depends on multiple regions

This notebook uses a local file upload so the same asset can be provided both as a model input and as a Code Interpreter container file.


In [ ]:
instructions = """
You are an expert document and chart analyst.

Use Code Interpreter whenever it helps.
For hard visual questions, zoom, crop, rotate, or inspect intermediate regions.
Prefer direct visual reasoning over OCR libraries unless OCR is clearly necessary.
"""

if RUN_LIVE_API:
    if not LOCAL_VISUAL_ASSET.exists():
        raise FileNotFoundError(f"Replace LOCAL_VISUAL_ASSET with a real file. Missing: {LOCAL_VISUAL_ASSET}")

    visual_file = client.files.create(
        file=LOCAL_VISUAL_ASSET.open("rb"),
        purpose="user_data",
    )

    ci_response = client.responses.create(
        model="gpt-5",
        instructions=instructions,
        tools=[
            {
                "type": "code_interpreter",
                "container": {
                    "type": "auto",
                    "memory_limit": "4g",
                    "file_ids": [visual_file.id],
                },
            }
        ],
        include=["code_interpreter_call.outputs"],
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "Read the attached visual carefully and identify the largest "
                            "drop between adjacent quarters. Explain which visual evidence "
                            "supports the answer."
                        ),
                    },
                    {
                        "type": "input_file",
                        "file_id": visual_file.id,
                    },
                ],
            }
        ],
    )

    print(ci_response.output_text)
else:
    print("Set RUN_LIVE_API = True and replace LOCAL_VISUAL_ASSET with a real file to run Code Interpreter.")


## Pattern 5: If you cannot use Code Interpreter, expose lightweight visual tools

In zero-data-retention or self-hosted environments, you may not want to give the model a general Python sandbox. In those cases, a small set of visual actions can recover much of the benefit:

- `crop_image`
- `zoom_image`
- `rotate_image`
- `ocr_region` (optional)

This is often cheaper and more controllable than unrestricted code execution while still giving the model a multi-pass visual workflow.


In [ ]:
custom_visual_tool_instructions = """
You are an expert document and image analyst.

Prefer direct visual reasoning first.
If a region is too small, rotated, or ambiguous, call crop_image, zoom_image, or rotate_image.
Only call ocr_region if direct visual reading is insufficient.
Explain what visual evidence you used before answering.
"""

recommended_visual_tools = [
    "crop_image",
    "zoom_image",
    "rotate_image",
    "ocr_region (optional)",
]

print({
    "recommended_visual_tools": recommended_visual_tools,
    "instructions_preview": custom_visual_tool_instructions,
})


## Pattern 6: Increase reasoning for chart, table, and form QA even without Python

Some multimodal tasks still improve if you increase reasoning effort, even when the model only sees the original image once. This is most useful when the answer depends on combining evidence across multiple regions of the page.

Reasoning tends to help more for:

- chart QA with small labels or multiple series
- table QA that requires connecting values across rows or columns
- menus, forms, and financial documents with table-like structure
- mixed-layout pages where the answer depends on several regions

Reasoning tends to help less for:

- pure transcription
- simple retrieval such as "what does paragraph X say?"
- needle-in-a-haystack lookups on otherwise readable pages

The tradeoff is higher token usage, so raise reasoning selectively.


In [ ]:
reasoning_only_prompt = """
Answer the question about the image.

Question:
Which quarter shows the largest drop from the previous quarter?

Instructions:
- Reason carefully across the full chart before answering.
- Cite the labels or values that support your answer.
- If the image is ambiguous, say what remains uncertain.
"""

if RUN_LIVE_API:
    reasoning_response = client.responses.create(
        model="gpt-5",
        reasoning={"effort": "high"},
        verbosity="high",
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": reasoning_only_prompt},
                    {
                        "type": "input_image",
                        "image_url": DOCUMENT_IMAGE_URL,
                        "detail": "high",
                    },
                ],
            }
        ],
    )
    print(reasoning_response.output_text)
else:
    print("Set RUN_LIVE_API = True to run the reasoning-without-Python experiment.")


## To Do: Compare models on the same Code Interpreter task

The easiest way to compare model behavior is to keep the prompt, asset, and tool configuration fixed. Then collect:

- the final answer
- whether the model actually used crop, zoom, or rotate behavior
- token usage
- qualitative failure modes

For GPT-5 series models, this notebook sets `verbosity="high"` to encourage more faithful, inspectable outputs.


In [ ]:
MODEL_MATRIX = [
    "gpt-4o",
    "gpt-5",
    "gpt-5.4",  # Placeholder: replace with the launch-approved slug before public release.
]

comparison_prompt = """
Read the attached visual carefully and answer the question below.

Question:
What is the largest drop between adjacent quarters?

Requirements:
- Use Code Interpreter if it helps.
- If the labels are hard to read, zoom or crop before answering.
- State the evidence you used.
"""

if RUN_LIVE_API:
    if not LOCAL_VISUAL_ASSET.exists():
        raise FileNotFoundError(f"Replace LOCAL_VISUAL_ASSET with a real file. Missing: {LOCAL_VISUAL_ASSET}")

    visual_file = client.files.create(
        file=LOCAL_VISUAL_ASSET.open("rb"),
        purpose="user_data",
    )

    results = {}
    for model in MODEL_MATRIX:
        request = {
            "model": model,
            "instructions": instructions,
            "tools": [
                {
                    "type": "code_interpreter",
                    "container": {
                        "type": "auto",
                        "memory_limit": "4g",
                        "file_ids": [visual_file.id],
                    },
                }
            ],
            "include": ["code_interpreter_call.outputs"],
            "input": [
                {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": comparison_prompt},
                        {"type": "input_file", "file_id": visual_file.id},
                    ],
                }
            ],
        }

        if model.startswith("gpt-5"):
            request["verbosity"] = "high"

        response = client.responses.create(**request)
        results[model] = {
            "response_id": response.id,
            "output_text": response.output_text,
            "usage": response.usage.model_dump() if response.usage else None,
        }

    results
else:
    print("Set RUN_LIVE_API = True to run the model comparison harness.")


## Placeholder: output comparison section

Replace this section with real excerpts before publishing.

| Model | Output excerpt | Tool behavior | Notes |
|---|---|---|---|
| `gpt-4o` | `[PASTE OUTPUT EXCERPT]` | `[Did it crop, zoom, or rotate?]` | `[Strengths, misses, or ambiguities]` |
| `gpt-5` | `[PASTE OUTPUT EXCERPT]` | `[Did it crop, zoom, or rotate?]` | `[Strengths, misses, or ambiguities]` |
| `gpt-5.4` | `[PASTE OUTPUT EXCERPT]` | `[Did it crop, zoom, or rotate?]` | `[Strengths, misses, or ambiguities]` |

## Placeholder: screenshot or crop comparison section

Replace this section with the exact artifacts you want to show in the final cookbook page.

- `[INSERT IMAGE: original chart or page]`
- `[INSERT IMAGE: gpt-4o intermediate crop or notable failure case]`
- `[INSERT IMAGE: gpt-5 intermediate crop or notable success case]`
- `[INSERT IMAGE: gpt-5.4 intermediate crop or notable success case]`


## Validation notes

Start with native vision when:

- the question is simple
- the page is not especially dense
- you care most about speed and cost

Add Code Interpreter when:

- the model needs multiple inspections
- the page is small, rotated, or cluttered
- you want the model to verify a reading instead of relying on one pass
- qualitative accuracy matters more than minimal token usage

Increase reasoning, even without Python, when:

- the answer depends on multiple regions of a chart, table, or form
- the task is analytical rather than pure transcription
- you can afford extra reasoning tokens for higher confidence

Expose lightweight visual tools instead of Code Interpreter when:

- you are operating in a restricted or self-hosted environment
- you want tighter control over what image transformations are allowed
- you only need crop, zoom, rotate, or an optional OCR-region fallback
